# Building a Calculator tool and let langchain  agent use for precise arithmetic :
## Instead of letting the model guest answers, the agent will delegate all math to a safe deterministic functions

### Keys take away: Any reliable function can be wrapped as lanchain tool, giving your agent a real, verifiable skills

In [1]:
# Install packages
!pip install -qU langchain langchain-google-genai google-generativeai==0.8.5

In [2]:
#  Set your Gemini API key\n AQ.Ab8RN6JYLjGEeCfbNQP_LxYuD-jYSS9wQirMdTyJnph9ST8HKg
import os
from getpass import getpass
if not os.environ.get("GOOGLE_API_KEY"
):
    os.environ["GOOGLE_API_KEY"] = getpass("Enter your Gemini API key: ")
print(os.environ.get("GOOGLE_API_KEY"))

AQ.Ab8RN6JYLjGEeCfbNQP_LxYuD-jYSS9wQirMdTyJnph9ST8HKg


In [3]:
import ast # for parsing expressions
import operator as op # for safe arithmetic operations 
from typing import Union  # for typ hints 

from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent 


c:\Users\sileymane.dianka\Documents\python\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:

 # Allowed operators
_OPS = {
    ast.Add: op.add, ast.Sub: op.sub, ast.Mult: op.mul,
    ast.Div: op.truediv, ast.FloorDiv: op.floordiv, ast.Mod: op.mod,
    ast.Pow: op.pow, ast.UAdd: op.pos, ast.USub: op.neg,
}

def _eval_node(node) -> Union[int, float]:
    # Python 3.8+ uses Constant for numbers
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_eval_node(node.left), _eval_node(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_eval_node(node.operand))
    if isinstance(node, ast.Expr):
        return _eval_node(node.value)
    raise ValueError("Unsupported expression. Use numbers, + - * / // % ** and parentheses only.")

def safe_calc(expr: str) -> float:
    expr = expr.strip()
    if len(expr) > 200:
        raise ValueError("Expression too long.")
    tree = ast.parse(expr, mode="eval")
    return float(_eval_node(tree.body))

@tool
def calculate(expression: str) -> str:
    """Safely evaluate arithmetic like '(12345*6789)+98765'. Returns only the number."""
    try:
        val = safe_calc(expression)
        if abs(val) > 1e18:
            return "Error: result too large."
        # Pretty output: drop trailing .0 if it's an integer
        return str(int(val)) if val.is_integer() else str(val)
    except Exception as e:
        return f"Error: {e}"


def build_calc_agent():
    llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash", temperature=0)
    tools = [calculate]
    prompt = ChatPromptTemplate.from_messages([
        ("system",
         "You are a precise math assistant.\n"
         "RULES:\n"
         "1) Always use the `calculate` tool for arithmetic; do not compute in your head.\n"
         "2) Convert natural language to a clean math expression.\n"
         "   - If user writes '^' for exponent, convert to '**'.\n"
         "3) If user asks for 'only the number', return only the number.\n"
         "4) Keep expressions short and safe."),
        ("human","{input}"),
        ("placeholder","{agent_scratchpad}"),
    ])
    agent = create_tool_calling_agent(llm, tools, prompt)
    return AgentExecutor(agent=agent, tools=tools, verbose=False)

calc_agent = build_calc_agent()
print(" Calculator agent ready.")


 Calculator agent ready.


In [8]:
print(calc_agent.invoke({"input": "Compute (12345*6789)+98765. Give only the number."})["output"])

c:\Users\sileymane.dianka\Documents\python\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3719: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Users\sileymane.dianka\Documents\python\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3719: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': '83908970', 'index': 0, 'extras': {'signature': 'EtIBCs8BARFNMg/kxyNLWDWxDguaI3LiVZVhnlYRhCq05pm4qei7uIfvXMvvy6Qi3T0ugmcyQ5L7gZoHOT/opxJvhJj+o4AEDUG+dGeq3c44yPo6AYMnzPQ84LhNqwBBekajz6x5EElxi334Eii8cjBYzSAddPDXCGaEVE3yoq3mctMD9R1951brVYxJGedE8k0wsjJYYe6u5UozuAtiZ8So53F04Zu1Xf3AAC65weWEjB2H2puz0kaHTUl4kFCV0bUNN41olG+FPyzdzTRB56HmpJhV'}}]


In [5]:
#Here we compare plain gemini (no tools) with with agent above: for comparison
llm_plain = ChatGoogleGenerativeAI(model="gemini-3.6-flash", temperature=0)

q1 = "What is 234 * 19? Give only the number."
q2 = "Compute (12345*6789)+98765. Give only the number."
q3 = "What is 13.7% of 12345 plus 98.76^2? Round to 2 decimals."

print("— LLM only —")
print("Q1:", llm_plain.invoke(q1).content)
print("Q2:", llm_plain.invoke(q2).content)
print("Q3:", llm_plain.invoke(q3).content)

print("\n— Agent with calculator —")
print("Q1:", calc_agent.invoke({"input": q1})["output"])
print("Q2:", calc_agent.invoke({"input": q2})["output"])
print("Q3:", calc_agent.invoke({"input": q3})["output"])

c:\Users\sileymane.dianka\Documents\python\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


— LLM only —
Q1: [{'type': 'text', 'text': '4446', 'extras': {'signature': 'EtgECtUEARFNMg8MsxSq99rZjw2nRcTqmDz5E6MMeu0GZl+fv8ze8eRges+o9Nt2/oSh0xG4BzUkQos+VXDjhIZjDOI6rJ9k39zfRIhOe+LnJzK7fqd0VguHnnrPMMN3t2uUqC3w65VZ+CESqubJ5GMEMPJ4POua20fV6d2PtSuCISBs1AJoegzgfAAxeavI9rJ1NpyD7wCnd+nqu28Z/7raf7rcVpxZCzow+CDQloIlCet8pJ70F/DqQ+rRAZ/RGSgFtOfyxFaQiELM2y/8M+3eqQNVH4djlSGjd8sjNCj55qEiKgoNJhkohdGfC5Sw/WgS6II9eH/Qd4huG6C52/bg544geb9wEo2fVCfQOzTKCRnKs73SSuiUZuH4Nf5fxjo+v1UNTgAxWn+sCn/QML8Htjcsdr0/Rk8gCJYopn1vGQsPX4AzToB/mMLRNUv77/XrcNkHKdxbdT0vS+vNHyb8eQ9nM24YNnqkIoVQtb6ZFcbc3d7JzXiU/TEV4z8VCtE3Bm8P1cQiiEysfEZuYbNw+RYMxgIUEHMzbMCYjTteETFCxAvpDKoP3dmM/94cTpnyAlNlpndEDkTdqNFNAVv6eC8TIjCnKU8hW2FPxqYcadjgQCELgHA0lNFLLDD0nNKdFIccSiC27r+nrs0rFzRzlJo9OsBpCN4t+yfH1eSonbetokX1c1ZFMKVhIjYTsFn2nXrceGQ5PsvHkjrqy72bUgu5kl3up/ljJAhmHQgjJbwzED3+0whFmoxePlwNejPY/tMEMu5fj6CXNl354IlxaRTWLn6INCkP'}}]


c:\Users\sileymane.dianka\Documents\python\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Q2: [{'type': 'text', 'text': '83908970', 'extras': {'signature': 'EpELCo4LARFNMg/l65phdBvHbCQXr6CWO3AgXetIo5qtqxRuJBuUwvJqamyGsn72PDLjVJ0pI67yM3yeedaOqlRD6mcLOXc7BnOSOIF5NddtS8skIUYNp0osue9kTY2KJle7ks1+orm71yuDBnDjvURq2qdHecwOTV9TmwQhmf0oKL/3WO8nDv7n3NEAsunMSLaDZWGySmDxQo0xl5RMf9ozV/JU69tkj0nGZekh1YFJwsnqNuHK5QbQ1X3mB/a08JCMioaEmCmMaSZ5+sIILppUkpCLY+t7nvvEZFnxtjLac9L20wvhea3867ydzDFjKJWqAj1UtMLuFxwIHW6y8ac1VOtXwRdXkEk5opVKrKfrluh688sPcwyhCW6RbkXlEDy+pXRt5YsOIo70HtqDsovHPCp+p4b1+ZRN1KKCy5U68X2kDqC9IJMobRCZhTQ/GCIs+58DR0D66FNR727EvXHJJG2L6PG+ye76pobjr3PV50FRZu6mT8cCN8RTDCSHAgTtCGP7H9SY/QZV1cLyxEr84L8JVWr8fGHVkrA71SI5y/9WS+kQ6JR5aH4jkKie38LQ6XCsjqH8ZkFCL+HZo+M2EW3sWGbjjKLuuaAJHnBNxGkXNggkHESHbHg3THo3D7RWHJJDanV0bxL4cXmsnxjms7m9xH3zcmIjyy8ffgDXaTDHXNDf0b43Uc/AAoBxczxmQsZ5lawHYB3s3dh+eLnot4eRdu1W6u4rxGstjXov+aciV/lWgT01HTubkHOtyx9UWvq+GPwmBtJjU7mwo09cJbObjudErZWBhnMVdZMEnmPoQ+5ScHpMk+ko+p42ZFGgSo8ybBv5S1kAEhL3SdfZoRLMk/yuEj/azQt1wSSp+Cfttlbs+EuU/E2BWTjmy84gQMraIlXhNyONoXqx2kfe8guuqKqTED1w1

c:\Users\sileymane.dianka\Documents\python\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Q3: [{'type': 'text', 'text': 'To find the answer, we can break the calculation down into two parts:\n\n1. **Calculate $13.7\\%$ of $12345$:**\n   $$\\text{Part 1} = 0.137 \\times 12345 = 1691.265$$\n\n2. **Calculate $98.76^2$:**\n   $$\\text{Part 2} = 98.76 \\times 98.76 = 9753.5376$$\n\n3. **Add the two results together:**\n   $$1691.265 + 9753.5376 = 11444.8026$$\n\n4. **Round to 2 decimal places:**\n   $$11444.8026 \\approx 11444.80$$\n\n**Final Answer:**\n**11,444.80**', 'extras': {'signature': 'EsoRCscRARFNMg/EbnrJnvv9j2a9e2kdj8wE39Hu9n67GZKobp/7/ZMI66XIIBW/zIwAjDpDFW3kOhCPE2vGae4Mvv/krTBNw1P8E4wDoZsF/t+xkZcTf9Z6UPcNj3cCZetKW1+QluOThhIIpc5TqPhqgthYTjv+bSu5wKsFkPwgJbW2ITyUuIhknfvfIq3LFkn49MmKuAh7i8CePKxEoMM8SzKkuy9bxv1VHVn13mI5UX2zWQ9itqZ8fJMlQlR596kc5TL4tdyMjYZRbdFRC2Jpr2I1ZIPljLc/ulUA2pf1HyyT5KDbkJmFm4j+iRXmhF+YG3vzlPwt26bO/mnrnFj7CdFRgvStTf7P6W8VbJVG1kUg5xWgChF3lnyEc3rgHvwV/i4vGFcQOfyaK/Ix7YMsAczs+3djF2uRl1D/HnVwGMwYpJhQlCeFSQhd6ivwDj4Uicmje0JVMpc8BYRzyaf87QrlXPGROEBnQUkZOIjWFk

c:\Users\sileymane.dianka\Documents\python\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\Users\sileymane.dianka\Documents\python\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Q1: [{'type': 'text', 'text': '4446', 'index': 0, 'extras': {'signature': 'EvABCu0BARFNMg81dfqq6kz7UXo4OvHKcuzYdbyiCTUNP+kOdBhxMdbDiiv2SsdpNdhMKeMfFnmxOReOqeiZxdPc3K1mrP3OWbg/wr6NlOX4mfgnqEwJ6rL9vUyKstqoozqctDGPTo0mD6/zZhwtcYFbKseegrHtC1y49Jj+ZRg04Lvr5cZxShUtOd9PK8fA2BFOqPzvJ+W9MYwJEfHCHcT4Nzx95HG1tAQiSaYpeAIkIE3yX8H+SdXz6u1hnWA27A8RjLDSm86euIALhOPyJgdCaPFFKtsK0cV/Et/44Olyegai3aX24ilPRocTHCKiqjIw'}}]


c:\Users\sileymane.dianka\Documents\python\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


GoogleRateLimitError: Error calling model 'gemini-3.6-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-3.6-flash\nPlease retry in 18.288935514s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerMinutePerProjectPerModel-FreeTier',